In [2]:
from pyspark.sql import functions as F

df_fact_bookings = spark.read.csv(
    "abfss://customersdata@adlshealthcaredata.dfs.core.windows.net/Fact_Bookings.csv",
    header=True
)

df_dim_property = spark.read.csv(
    "abfss://customersdata@adlshealthcaredata.dfs.core.windows.net/Dim_Property.csv",
    header=True
)

df_dim_customer = spark.read.csv(
    "abfss://customersdata@adlshealthcaredata.dfs.core.windows.net/Dim_Customer.csv",
    header=True
)

df_dim_host = spark.read.csv(
    "abfss://customersdata@adlshealthcaredata.dfs.core.windows.net/Dim_Host.csv",
    header=True
)

df_dim_date = spark.read.csv(
    "abfss://customersdata@adlshealthcaredata.dfs.core.windows.net/Dim_Date.csv",
    header=True
)

StatementMeta(sparkpool, 5, 3, Finished, Available, Finished, False)

In [17]:
#df_fact_bookings.columns
#df_dim_property.columns
#df_dim_customer.columns
#df_dim_host.columns
df_dim_date.columns

StatementMeta(sparkpool, 5, 18, Finished, Available, Finished, False)

['Date_ID',
 'Full_Date',
 'Day',
 'Month',
 'Quarter',
 'Year',
 'Weekday_Name',
 'Is_Weekend']

In [18]:
df_fact_bookings.printSchema()

df_fact_bookings.show(5)

StatementMeta(sparkpool, 5, 19, Finished, Available, Finished, False)

root
 |-- Booking_ID: string (nullable = true)
 |-- Property_ID: string (nullable = true)
 |-- Date_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Host_ID: string (nullable = true)
 |-- Revenue: string (nullable = true)
 |-- Nights_Booked: string (nullable = true)
 |-- Number_of_Guests: string (nullable = true)
 |-- Cleaning_Fee: string (nullable = true)
 |-- Service_Fee: string (nullable = true)

+----------+-----------+-------+-----------+-------+-------+-------------+----------------+------------+-----------+
|Booking_ID|Property_ID|Date_ID|Customer_ID|Host_ID|Revenue|Nights_Booked|Number_of_Guests|Cleaning_Fee|Service_Fee|
+----------+-----------+-------+-----------+-------+-------+-------------+----------------+------------+-----------+
|         0|        789|   1159|       3090|    135|3213.39|           18|               5|      190.09|      121.4|
|         1|        341|    762|       2579|     75| 205.34|           19|               6|       56.

In [19]:
df_fact_bookings = df_fact_bookings.dropDuplicates()

StatementMeta(sparkpool, 5, 20, Finished, Available, Finished, False)

In [20]:
df_fact_bookings = df_fact_bookings.fillna({
    "Cleaning_Fee": 0,
    "Service_Fee": 0
})

StatementMeta(sparkpool, 5, 21, Finished, Available, Finished, False)

Convert Data Types -revenue

In [21]:
df_fact_bookings = df_fact_bookings.withColumn(
    "Revenue",
    F.col("Revenue").cast("double")
)

StatementMeta(sparkpool, 5, 22, Finished, Available, Finished, False)

nights_booked

In [22]:
df_fact_bookings = df_fact_bookings.withColumn(
    "Nights_Booked",
    F.col("Nights_Booked").cast("integer")
)

StatementMeta(sparkpool, 5, 23, Finished, Available, Finished, False)

Number_of_Guests

In [23]:
df_fact_bookings = df_fact_bookings.withColumn(
    "Number_of_Guests",
    F.col("Number_of_Guests").cast("integer")
)

StatementMeta(sparkpool, 5, 24, Finished, Available, Finished, False)

Date Conversion

In [24]:
df_dim_date = df_dim_date.withColumn(
    "Full_Date",
    F.to_date(F.col("Full_Date"), "yyyy-MM-dd")
)

StatementMeta(sparkpool, 5, 25, Finished, Available, Finished, False)

**Wide Transformations (JOINS) fact +dimensions**

customer analytics dataframe- booking + customer + date analytics

In [41]:
df_customer_analytics = df_fact_bookings \
    .join(df_dim_customer, "Customer_ID") \
    .join(df_dim_date, "Date_ID")

StatementMeta(sparkpool, 5, 42, Finished, Available, Finished, False)

Property Analytics DataFrame

In [75]:
df_property_analytics_clean = (
    df_fact_bookings
    .select("Booking_ID", "Property_ID", "Host_ID", "Revenue")
    .join(
        df_dim_property.select(
            "Property_ID",
            "Property_Type",
            "Location_City",
            "Location_Country",
            "Price_per_Night"
        ),
        "Property_ID"
    )
    .join(
        df_dim_host.select(
            "Host_ID",
            "Host_Name",
            "Superhost_Flag",
            "Total_Listings"
        ),
        "Host_ID"
    )
)

StatementMeta(sparkpool, 5, 76, Finished, Available, Finished, False)

GROUP BY Aggregations

Revenue by Country

In [69]:
df_country_revenue = df_property_analytics \
    .groupBy("Location_Country") \
    .agg(
        F.sum("Revenue").alias("Total_Revenue")
    )

StatementMeta(sparkpool, 5, 70, Finished, Available, Finished, False)

Revenue by Customer Segment

In [70]:
df_customer_segment = df_customer_analytics \
    .groupBy("Customer_Segment") \
    .agg(
        F.sum("Revenue").alias("Total_Revenue")
    )

StatementMeta(sparkpool, 5, 71, Finished, Available, Finished, False)

monthly booking trend

In [71]:
df_monthly_bookings = df_customer_analytics \
    .groupBy("Year", "Month") \
    .agg(
        F.sum("Revenue").alias("Monthly_Revenue"),
        F.count("Booking_ID").alias("Total_Bookings")
    )

StatementMeta(sparkpool, 5, 72, Finished, Available, Finished, False)

Write DataFrames to Delta Tables

Customer Analytics Delta Table

In [82]:
df_customer_analytics.write \
    .format("delta") \
    .mode("overwrite") \
    .save(
        "abfss://customersdata@adlshealthcaredata.dfs.core.windows.net/delta/customer_analytics"
    )

StatementMeta(sparkpool, 5, 83, Finished, Available, Finished, False)

In [76]:
df_property_analytics_clean.printSchema()

StatementMeta(sparkpool, 5, 77, Finished, Available, Finished, False)

root
 |-- Host_ID: string (nullable = true)
 |-- Property_ID: string (nullable = true)
 |-- Booking_ID: string (nullable = true)
 |-- Revenue: double (nullable = true)
 |-- Property_Type: string (nullable = true)
 |-- Location_City: string (nullable = true)
 |-- Location_Country: string (nullable = true)
 |-- Price_per_Night: string (nullable = true)
 |-- Host_Name: string (nullable = true)
 |-- Superhost_Flag: string (nullable = true)
 |-- Total_Listings: string (nullable = true)



Property Analytics Delta Table

In [78]:
df_property_analytics_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .save(
        "abfss://customersdata@adlshealthcaredata.dfs.core.windows.net/delta/property_analytics"
    )

StatementMeta(sparkpool, 5, 79, Finished, Available, Finished, False)

FILTER FOR TABLE 1 (Customer Analytics)-high revenue + recent bookings

In [79]:
df_filter1 = df_customer_analytics.filter(
    (F.col("Revenue") > 500) &
    (F.col("Year") >= 2024)
)

StatementMeta(sparkpool, 5, 80, Finished, Available, Finished, False)

FILTER FOR TABLE 2 (Property Analytics)
superhost + high price properties

In [80]:
df_filter2 = df_property_analytics_clean.filter(
    (F.col("Superhost_Flag") == "Yes") &
    (F.col("Price_per_Night") > 200)
)

StatementMeta(sparkpool, 5, 81, Finished, Available, Finished, False)

In [86]:
from delta.tables import DeltaTable

StatementMeta(sparkpool, 5, 87, Finished, Available, Finished, False)

Customer Analytics Delta Table

In [87]:
delta_customer_analytics = DeltaTable.forPath(
    spark,
    "abfss://customersdata@adlshealthcaredata.dfs.core.windows.net/delta/customer_analytics"
)

StatementMeta(sparkpool, 5, 88, Finished, Available, Finished, False)

Property Analytics Delta Table

In [88]:
delta_property_analytics = DeltaTable.forPath(
    spark,
    "abfss://customersdata@adlshealthcaredata.dfs.core.windows.net/delta/property_analytics"
)

StatementMeta(sparkpool, 5, 89, Finished, Available, Finished, False)

UPSERT df_filter1 to customer_analytics
Key = Booking_ID

In [89]:
delta_customer_analytics.alias("t").merge(
    df_filter1.alias("s"),
    "t.Booking_ID = s.Booking_ID"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

StatementMeta(sparkpool, 5, 90, Finished, Available, Finished, False)

UPSERT df_filter2 to property_analytics
Key = Property_ID

In [90]:
delta_property_analytics.alias("t").merge(
    df_filter2.alias("s"),
    "t.Property_ID = s.Property_ID"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

StatementMeta(sparkpool, 5, 91, Finished, Available, Finished, False)

writting to dedicatedsqlpool

In [127]:
jdbc_url = "jdbc:sqlserver://arvindwork.sql.azuresynapse.net:1433;database=dedicatedsqlpool"
connection_properties = {
    "user": "sqladminuser",
    "password": "Aravind@123",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

StatementMeta(sparkpool, 5, 128, Finished, Available, Finished, False)

In [116]:
from pyspark.sql.functions import col

df_customer_analytics_fixed = df_customer_analytics \
    .withColumn("Date_ID", col("Date_ID").cast("string"))

StatementMeta(sparkpool, 5, 117, Finished, Available, Finished, False)

In [121]:
df_property_analytics.columns

StatementMeta(sparkpool, 5, 122, Finished, Available, Finished, False)

['Property_ID',
 'Booking_ID',
 'Date_ID',
 'Customer_ID',
 'Host_ID',
 'Revenue',
 'Nights_Booked',
 'Number_of_Guests',
 'Cleaning_Fee',
 'Service_Fee',
 'Host_ID',
 'Property_Type',
 'Room_Type',
 'Location_City',
 'Location_Country',
 'No_of_Bedrooms',
 'No_of_Bathrooms',
 'Amenities',
 'Price_per_Night',
 'Host_Name',
 'Superhost_Flag',
 'Total_Listings',
 'Host_Since_Year',
 'Response_Time']

In [124]:
df_customer_analytics.write \
    .mode("append") \
    .jdbc(url=jdbc_url, table="dbo.customer_analytics", properties=connection_properties)

StatementMeta(sparkpool, 5, 125, Finished, Available, Finished, False)

In [130]:
df_property_analytics_clean.write \
    .mode("append") \
    .jdbc(url=jdbc_url, table="dbo.property_analytics", properties=connection_properties)

StatementMeta(sparkpool, 5, 131, Finished, Available, Finished, False)